# DeBERTa v3 on SNLI

Colab-ready fine-tuning notebook with a GPU runtime expected. The code stays explicit so you can inspect the validation loss, accuracy, and saved artifacts directly.


In [1]:
%pip install -q transformers datasets torch accelerate sentencepiece evaluate scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


In [2]:
import json
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

print("GPU available:", torch.cuda.is_available())

GPU available: False


In [3]:
MODEL_NAME = "microsoft/deberta-v3-small"
OUTPUT_DIR = Path("deberta_snli_colab")
MAX_TRAIN_SAMPLES = 15000
MAX_EVAL_SAMPLES = 2000
MAX_LENGTH = 256
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

print(MODEL_NAME)
print("Output dir:", OUTPUT_DIR)

microsoft/deberta-v3-small
Output dir: deberta_snli_colab


In [4]:
dataset = load_dataset("snli")
dataset = dataset.filter(lambda row: row["label"] != -1)

label_names = dataset["train"].features["label"].names
id2label = {idx: name for idx, name in enumerate(label_names)}
label2id = {name: idx for idx, name in id2label.items()}

train_split = dataset["train"].shuffle(seed=SEED).select(range(min(MAX_TRAIN_SAMPLES, len(dataset["train"]))))
eval_split = dataset["validation"].select(range(min(MAX_EVAL_SAMPLES, len(dataset["validation"]))))

train_label_counts = Counter(train_split["label"])
majority_label_id = train_label_counts.most_common(1)[0][0]
majority_baseline = sum(int(label == majority_label_id) for label in eval_split["label"]) / len(eval_split)

print("Label names:", label_names)
print("Train size:", len(train_split))
print("Validation size:", len(eval_split))
print("Train label counts:", {id2label[idx]: count for idx, count in sorted(train_label_counts.items())})
print("Majority baseline:", round(majority_baseline, 4))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/412k [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/413k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/19.6M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/550152 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/550152 [00:00<?, ? examples/s]

Label names: ['entailment', 'neutral', 'contradiction']
Train size: 15000
Validation size: 2000
Train label counts: {'entailment': 4970, 'neutral': 4984, 'contradiction': 5046}
Majority baseline: 0.33


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
print(tokenizer.__class__.__name__)

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

DebertaV2Tokenizer


In [6]:
def tokenize_function(batch):
    return tokenizer(
        batch["premise"],
        batch["hypothesis"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_tokenized = train_split.map(tokenize_function, batched=True)
eval_tokenized = eval_split.map(tokenize_function, batched=True)

train_tokenized = train_tokenized.rename_column("label", "labels").remove_columns(["premise", "hypothesis"])
eval_tokenized = eval_tokenized.rename_column("label", "labels").remove_columns(["premise", "hypothesis"])

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)
model.config.problem_type = "single_label_classification"

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias       

In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "macro_f1": float(f1_score(labels, predictions, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(labels, predictions, average="weighted", zero_division=0)),
    }

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    fp16=False,  # Keep training stable; mixed precision is not needed here.
    dataloader_num_workers=0,
    seed=SEED,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [10]:
train_result = trainer.train()
print(train_result)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
eval_metrics = trainer.evaluate()
prediction_output = trainer.predict(eval_tokenized)

logits = prediction_output.predictions
predicted_ids = np.argmax(logits, axis=-1)
probabilities = torch.softmax(torch.tensor(logits), dim=-1).numpy()
predicted_confidence = probabilities.max(axis=-1)

predicted_labels = [id2label[int(label_id)] for label_id in predicted_ids]
true_labels = [id2label[int(label_id)] for label_id in eval_split["label"]]

report = classification_report(
    true_labels,
    predicted_labels,
    labels=label_names,
    zero_division=0,
    output_dict=True,
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
predictions_path = OUTPUT_DIR / "eval_predictions.csv"
summary_path = OUTPUT_DIR / "summary.json"

pd.DataFrame(
    {
        "premise": eval_split["premise"],
        "hypothesis": eval_split["hypothesis"],
        "true_label": true_labels,
        "pred_label": predicted_labels,
        "pred_confidence": predicted_confidence,
    }
).to_csv(predictions_path, index=False)

trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

summary = {
    "model_name": MODEL_NAME,
    "seed": SEED,
    "data": {
        "train_samples": len(train_split),
        "eval_samples": len(eval_split),
        "label_names": label_names,
        "train_label_counts": {id2label[idx]: int(count) for idx, count in sorted(train_label_counts.items())},
        "majority_label": id2label[majority_label_id],
        "majority_baseline_accuracy": float(majority_baseline),
    },
    "training": {
        "train_metrics": train_result.metrics,
    },
    "evaluation": {
        "eval_metrics": eval_metrics,
        "prediction_metrics": prediction_output.metrics,
        "report": report,
    },
    "artifacts": {
        "predictions_csv": str(predictions_path),
        "summary_json": str(summary_path),
        "model_dir": str(OUTPUT_DIR),
    },
}

summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=True), encoding="utf-8")

print("Eval metrics:", eval_metrics)
print("Predictions saved to:", predictions_path)
print("Summary saved to:", summary_path)